# 02 - 特征工程与训练数据构建

**目标**: 从光变曲线批量提取特征, 构建可用于 ML 的训练/验证/测试集。

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tqdm.notebook import tqdm
import pickle
import os

%matplotlib inline

## 1. 加载数据

In [ ]:
from src.data.download import load_plasticc
from src.data.preprocess import CLASS_NAMES, extract_lightcurve_by_object
from src.features.extractor import extract_features, extract_features_batch

train_df, test_df, meta = load_plasticc("../data/raw/plasticc")

print(f"训练集天体数: {meta['object_id'].nunique() if meta is not None else 'N/A'}")

## 2. 构建特征矩阵

对每个天体的 r-band (passband=3) 光变曲线提取统计特征。
由于 PLAsTiCC 数据量很大 (百万级), 我们先取一个子集用于快速实验。

In [ ]:
# 每个类别取 N 个样本
SAMPLES_PER_CLASS = 200
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

if meta is not None:
    all_objects = []
    for cls_id in sorted(meta["target"].unique()):
        obj_ids = meta[meta["target"] == cls_id]["object_id"].values
        n_take = min(SAMPLES_PER_CLASS, len(obj_ids))
        chosen = np.random.choice(obj_ids, size=n_take, replace=False)
        for oid in chosen:
            all_objects.append((oid, cls_id))

    print(f"共选取 {len(all_objects)} 个天体")
else:
    print("未找到 metadata, 请先下载 PLAsTiCC 数据集")

In [ ]:
# 批量提取特征 (这一步可能较慢, 取决于样本量)
features_list = []
labels_list = []
skipped = 0

for obj_id, cls_id in tqdm(all_objects, desc="提取特征"):
    obj = train_df[train_df["object_id"] == obj_id]
    band = obj[obj["passband"] == 3]  # r-band

    if len(band) < 5:  # 至少需要 5 个观测点
        skipped += 1
        continue

    feat = extract_features(
        band["mjd"].values,
        band["flux"].values,
        band["flux_err"].values
    )
    features_list.append(feat)
    labels_list.append(cls_id)

print(f"提取成功: {len(features_list)}, 跳过 (点数不足): {skipped}")

X = pd.DataFrame(features_list)
y = pd.Series(labels_list, name="target")

print(f"\n特征矩阵: {X.shape}")
print(f"标签分布:\n{y.value_counts().sort_index()}")

## 3. 数据清洗与标准化

In [ ]:
# 处理无穷值和 NaN
X = X.replace([np.inf, -np.inf], np.nan)

# 检查每列缺失比例
missing_ratio = X.isnull().mean()
print("各列缺失比例:")
print(missing_ratio[missing_ratio > 0])

# 用中位数填充缺失值
X = X.fillna(X.median())

# 去除方差为 0 的列
zero_var_cols = X.columns[X.std() == 0]
if len(zero_var_cols) > 0:
    print(f"去除零方差列: {list(zero_var_cols)}")
    X = X.drop(columns=zero_var_cols)

print(f"清洗后特征维度: {X.shape}")

In [ ]:
# 标签编码: 将原始 class_id 映射为 0..N-1
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("标签映射:")
for i, cls_id in enumerate(label_encoder.classes_):
    name = CLASS_NAMES.get(cls_id, f"class_{cls_id}")
    print(f"  {i:3d} -> {cls_id:4d} ({name})")

In [ ]:
# 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f"标准化后: mean={X_scaled.mean().mean():.4f}, std={X_scaled.std().mean():.4f}")

## 4. 划分数据集

In [ ]:
# 分层抽样: 训练 70% / 验证 15% / 测试 15%
X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.15,
    stratify=y_encoded,
    random_state=RANDOM_SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.15 / 0.85,  # 15% 的原始数据
    stratify=y_temp,
    random_state=RANDOM_SEED
)

print(f"训练集: {X_train.shape[0]} 样本")
print(f"验证集: {X_val.shape[0]} 样本")
print(f"测试集: {X_test.shape[0]} 样本")
# 验证分层
print(f"\n训练集类别分布:\n{pd.Series(y_train).value_counts().sort_index()}")

## 5. 保存处理后的数据

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

# 保存 NumPy 数组
np.savez(
    "../data/processed/train.npz",
    X=X_train.values.astype(np.float32),
    y=y_train.astype(np.int32)
)
np.savez(
    "../data/processed/val.npz",
    X=X_val.values.astype(np.float32),
    y=y_val.astype(np.int32)
)
np.savez(
    "../data/processed/test.npz",
    X=X_test.values.astype(np.float32),
    y=y_test.astype(np.int32)
)

# 保存 scaler 和 label_encoder
with open("../data/processed/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("../data/processed/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("数据已保存到 ../data/processed/")

## 6. 特征重要性初探

用简单的单变量分析看哪些特征与目标最相关。

In [ ]:
from sklearn.feature_selection import mutual_info_classif

mi = mutual_info_classif(X_train, y_train, random_state=RANDOM_SEED)
mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
mi_series.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Mutual Information")
ax.set_title("特征与目标变量的互信息")
plt.tight_layout()
plt.show()

print("Top 5 特征:")
print(mi_series.head(5))

## 7. 下一步

- **03_baseline_model**: 用 XGBoost 训练基线分类器